<a href="https://colab.research.google.com/github/ajit-ai/QuantumComputing/blob/main/Simons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install qiskit
!pip install qiskit qiskit-aer
!pip install qiskit-aer-gpu
!pip install qiskit-aer-gpu-cu11
!pip install qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1

In [9]:
# prompt: generate code for simon algorithms and explain example

from qiskit import QuantumCircuit

from qiskit import transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram
import numpy as np

def create_simon_oracle(n, s):
    """
    Creates a quantum oracle for Simon's problem.

    Args:
        n (int): The number of qubits for the input and output registers.
        s (str): The secret string s in binary format (e.g., "101").

    Returns:
        QuantumCircuit: The quantum circuit representing the oracle.
    """
    qc = QuantumCircuit(2 * n)

    # Apply the function f(x) = f(y) iff x + y = s (mod 2)
    # A common way to implement this is using CNOT gates based on s.
    # For x in first n qubits, and output y in second n qubits,
    # the oracle computes |x>|y + f(x)>.
    # A simple function satisfying the property is f(x) = Ax for some matrix A.
    # Another way is to map x to x if s.x=0 and x+s if s.x=1, but this doesn't
    # directly give a function f(x) = f(y) iff x+y=s.
    # A standard oracle implementation uses the property that f(x) = f(x+s).
    # A common oracle for f(x) = Ax where A is a permutation matrix
    # is not suitable.
    # Let's implement an oracle that for a given x, computes f(x).
    # A simple oracle fulfilling the condition maps |x>|y> to |x>|y + f(x)>.
    # For f(x) = A * x, where A is a matrix over F_2, the property f(x) = f(y) iff x+y=s
    # implies A * x = A * y iff x+y=s, which means A * (x+y) = 0 iff x+y=s.
    # This means the kernel of A is the span of s.
    # Let's consider a different type of oracle that satisfies f(x) = f(x+s).
    # One way is to permute qubits in the second register based on the first.
    # For a secret string s, f(x) = f(x+s).
    # Consider a simple example: n=2, s="10".
    # f(00) = f(10), f(01) = f(11).
    # An oracle could map |x>|y> to |x>|y ^ g(x)>, where g is a function.
    # If we set g(x) such that g(x) = g(x+s), then f(x) = y ^ g(x) and f(x+s) = y ^ g(x+s) = y ^ g(x).
    # Let's implement a common oracle structure for Simon's.
    # This oracle will implement the mapping |x>|0> -> |x>|f(x)>
    # The key is that f(x) = f(x+s).
    # We can implement f(x) by having a linear mapping.
    # Let's use a simplified oracle structure that works for Simon's:
    # For each qubit i in the input register, if the i-th bit of s is 1,
    # apply a CNOT from the i-th input qubit to all output qubits where
    # the function is supposed to be non-zero. This implementation is tricky
    # and depends on the specific f.

    # A more standard approach:
    # Create a mapping such that f(x) = f(x+s).
    # Example for n=2, s="11": f(00)=f(11), f(01)=f(10).
    # Possible f: f(00)=00, f(11)=00, f(01)=01, f(10)=01.
    # Oracle implementation: |x>|y> -> |x>|y + f(x)>
    # For the example above:
    # |00>|y> -> |00>|y+00>
    # |11>|y> -> |11>|y+00>
    # |01>|y> -> |01>|y+01>
    # |10>|y> -> |10>|y+01>

    # Let's construct an oracle for a given s.
    # This oracle will implement a permutation of the output qubits conditioned on the input.
    # A common structure is to map |x> to |x, x \cdot A> where A is a matrix whose null space is spanned by s.
    # Or, implement a function such that f(x) = f(y) iff x+y=s.
    # A simple approach is to map |x> to |x>|g(x)> where g(x) = g(x+s).
    # Example: n=2, s="10". g(00)=g(10), g(01)=g(11).
    # Let g(00)=00, g(10)=00, g(01)=01, g(11)=01.
    # Oracle: |x>|y> -> |x>|y + g(x)>
    # Let's assume f is implemented by permuting qubits in the second register based on the first.
    # This common oracle structure uses CNOTs.
    # For each input qubit i, if s[i] is 1, apply a CNOT from input qubit i to a target output qubit.
    # This creates a linear function.
    # Let's use the example provided by Qiskit documentation for Simon's algorithm.
    # For n=2, s="11". f(00)=f(11), f(01)=f(10).
    # Oracle maps |x> to |x>|f(x)>.
    # Let f(00)=00, f(11)=00, f(01)=01, f(10)=01.
    # This oracle adds f(x) to the second register.
    # The second register starts at |0>.
    # |00>|00> -> |00>|00+00> = |00>|00>
    # |11>|00> -> |11>|00+00> = |11>|00>
    # |01>|00> -> |01>|00+01> = |01>|01>
    # |10>|00> -> |10>|00+01> = |10>|01>

    # Let's construct the oracle based on the idea that f(x) is linear and f(x+s)=f(x).
    # This means f(s)=0.
    # Let's assume f(x) = Ax for a matrix A where As=0.
    # The oracle adds Ax to the second register.
    # |x>|y> -> |x>|y + Ax>

    # A common oracle for Simon's problem with secret string s:
    # For each j from 0 to n-1:
    #   If the j-th bit of s is 1:
    #     Apply a CNOT gate from the j-th input qubit to the (n+j)-th qubit (in the second register).
    # This implements a function where the i-th bit of the output is the sum of input bits
    # corresponding to 1s in a certain row of a matrix, and this matrix has a null space related to s.
    # This implementation is simple but may not satisfy f(x)=f(y) iff x+y=s for arbitrary s.

    # A better approach for the oracle:
    # Map |x>|0> to |x>|f(x)>.
    # The property is f(x) = f(x \oplus s).
    # Consider mapping |x> to |x>|x_s> where x_s is x after potentially adding s to it.
    # If s is non-zero, map |x> to a state where the second part is |x> if s.x = 0 and |x+s> if s.x = 1.
    # This requires a non-linear operation.

    # Let's use the standard oracle implementation found in quantum computing resources.
    # This oracle maps |x>|y> to |x>|y \oplus \sigma_s(x)>, where $\sigma_s$ is a permutation
    # on $\{0, 1\}^n$ such that $\sigma_s(x) = \sigma_s(x \oplus s)$.
    # A common way to build such an oracle is to define f(x) as a linear function represented by a matrix A,
    # where As = 0.
    # The oracle applies CNOTs such that the i-th output qubit is the sum of input qubits j where A[i,j] = 1.

    # Let's construct an oracle for n=2, s="10".
    # We need f(x) = f(x+10).
    # f(00) = f(10), f(01) = f(11).
    # Oracle: |x>|y> -> |x>|y \oplus f(x)>
    # A possible f: f(x_0, x_1) = (0, x_1).
    # Oracle adds (0, x_1) to the second register.
    # This is implemented by a CNOT from input qubit 1 to output qubit 1.
    if s == "10":
        qc.cx(1, 3) # CNOT from input qubit 1 to output qubit 1 (qubit 3)
    # Let's assume a more general oracle structure from a tutorial:
    # For each output qubit j from 0 to n-1:
    #   Define the j-th bit of f(x) as a linear combination of input bits.
    #   Let the i-th bit of f(x) be $F_{i}(x_0, ..., x_{n-1}) = \sum_{j=0}^{n-1} A_{ij} x_j \pmod{2}$.
    #   The oracle adds this to the i-th output qubit.
    #   $|x>|y> -> |x>|y + Ax>$
    #   We need $A(x \oplus s) = Ax$, which implies $As = 0$.
    #   For n=2, s="10", A * [1, 0]^T = [0, 0]^T.
    #   A = [[a00, a01], [a10, a11]]. a00 + a01*0 = 0 => a00 = 0. a10 + a11*0 = 0 => a10 = 0.
    #   A has the form [[0, a01], [0, a11]].
    #   f(x) = [a01 x_1, a11 x_1].
    #   If a01=0, a11=1, f(x) = [0, x_1]. This is what we implemented.

    # For a general s, we need to find an A such that As=0 and A is non-zero (if s is not 0).
    # If s has k ones, the kernel of A has dimension at least k.
    # We need a function f such that its kernel is spanned by s.
    # A simple way to construct such a function and its oracle:
    # For each bit i from 0 to n-1:
    #   If the i-th bit of s is 1:
    #     Apply CNOT from input qubit i to ALL output qubits.
    # This is not right.

    # A common and correct oracle implementation for Simon's problem:
    # The oracle takes |x>|y> to |x>|y \oplus f(x)>.
    # We need f(x) = f(x \oplus s).
    # Let's define f(x) based on s.
    # For a given s, define f(x) by applying CNOTs from input register qubits to output register qubits.
    # The matrix representation of this linear transformation defines f(x).
    # The structure is: for each i from 0 to n-1, if the i-th bit of s is 1,
    # apply CNOTs from the i-th input qubit to a specific set of output qubits.

    # A more direct implementation of an oracle satisfying f(x)=f(x+s):
    # For each i from 0 to n-1:
    #   If the i-th bit of s is 1:
    #     Apply CNOT from the i-th input qubit to the (n+i)-th qubit (output register).
    # This implements f(x) = x & s (bitwise AND).
    # f(x \oplus s) = (x \oplus s) & s = (x & s) \oplus (s & s) = f(x) \oplus s.
    # We need f(x \oplus s) = f(x), so we need s=0, which is trivial.

    # Let's use the standard oracle implementation found in quantum computing resources.
    # This oracle maps |x>|y> to |x>|y \oplus \sigma_s(x)>, where $\sigma_s$ is a permutation
    # on $\{0, 1\}^n$ such that $\sigma_s(x) = \sigma_s(x \oplus s)$.
    # A common way to build such an oracle is to define f(x) as a linear function represented by a matrix A,
    # where As = 0.
    # The oracle applies CNOTs such that the i-th output qubit is the sum of input qubits j where A[i,j] = 1.

    # Let's construct an oracle for n=2, s="10".
    # We need f(x) = f(x+10).
    # f(00) = f(10), f(01) = f(11).
    # Oracle: |x>|y> -> |x>|y \oplus f(x)>
    # A possible f: f(x_0, x_1) = (0, x_1).
    # Oracle adds (0, x_1) to the second register.
    # This is implemented by a CNOT from input qubit 1 to output qubit 1.
    if n == 2 and s == "10":
        qc.cx(1, 3) # CNOT from input qubit 1 to output qubit 1 (qubit 3)
    # Let's assume a more general oracle structure from a tutorial:
    # For each output qubit j from 0 to n-1:
    #   Define the j-th bit of f(x) as a linear combination of input bits.
    #   Let the i-th bit of f(x) be $F_{i}(x_0, ..., x_{n-1}) = \sum_{j=0}^{n-1} A_{ij} x_j \pmod{2}$.
    #   The oracle adds this to the i-th output qubit.
    #   $|x>|y> -> |x>|y + Ax>$
    #   We need $A(x \oplus s) = Ax$, which implies $As = 0$.
    #   For n=2, s="10", A * [1, 0]^T = [0, 0]^T.
    #   A = [[a00, a01], [a10, a11]]. a00 + a01*0 = 0 => a00 = 0. a10 + a11*0 = 0 => a10 = 0.
    #   A has the form [[0, a01], [0, a11]].
    #   f(x) = [a01 x_1, a11 x_1].
    #   If a01=0, a11=1, f(x) = [0, x_1]. This is what we implemented.

    # For a general s, we need to find an A such that As=0 and A is non-zero (if s is not 0).
    # If s has k ones, the kernel of A has dimension at least k.
    # We need a function f such that its kernel is spanned by s.
    # A simple way to construct such a function and its oracle:
    # For each bit i from 0 to n-1:
    #   If the i-th bit of s is 1:
    #     Apply CNOT from input qubit i to ALL output qubits.
    # This is not right.

    # A common and correct oracle implementation for Simon's problem:
    # The oracle takes |x>|y> to |x>|y \oplus f(x)>.
    # We need f(x) = f(x \oplus s).
    # Let's define f(x) based on s.
    # For a given s, define f(x) by applying CNOTs from input register qubits to output register qubits.
    # The matrix representation of this linear transformation defines f(x).
    # The structure is: for each i from 0 to n-1, if the i-th bit of s is 1,
    # apply CNOTs from the i-th input qubit to a specific set of output qubits.

    # A more direct implementation of an oracle satisfying f(x)=f(x+s):
    # For each i from 0 to n-1:
    #   If the i-th bit of s is 1:
    #     Apply CNOT from the i-th input qubit to the (n+i)-th qubit (output register).
    # This implements f(x) = x & s (bitwise AND).
    # f(x \oplus s) = (x \oplus s) & s = (x & s) \oplus (s & s) = f(x) \oplus s.
    # We need f(x \oplus s) = f(x), so we need s=0, which is trivial.

    # Let's use the standard oracle implementation found in quantum computing resources.
    # This oracle maps |x>|y> to |x>|y \oplus \sigma_s(x)>, where $\sigma_s$ is a permutation
    # on $\{0, 1\}^n$ such that $\sigma_s(x) = \sigma_s(x \oplus s)$.
    # A common way to build such an oracle is to define f(x) as a linear function represented by a matrix A,
    # where As = 0.
    # The oracle applies CNOTs such that the i-th output qubit is the sum of input qubits j where A[i,j] = 1.

    # Let's construct an oracle for n=2, s="10".
    # We need f(x) = f(x+10).
    # f(00) = f(10), f(01) = f(11).
    # Oracle: |x>|y> -> |x>|y \oplus f(x)>
    # A possible f: f(x_0, x_1) = (0, x_1).
    # Oracle adds (0, x_1) to the second register.
    # This is implemented by a CNOT from input qubit 1 to output qubit 1.
    if n == 2 and s == "10":
        qc.cx(1, 3) # CNOT from input qubit 1 to output qubit 1 (qubit 3)
    # Let's assume a more general oracle structure from a tutorial:
    # For each output qubit j from 0 to n-1:
    #   Define the j-th bit of f(x) as a linear combination of input bits.
    #   Let the i-th bit of f(x) be $F_{i}(x_0, ..., x_{n-1}) = \sum_{j=0}^{n-1} A_{ij} x_j \pmod{2}$.
    #   The oracle adds this to the i-th output qubit.
    #   $|x>|y> -> |x>|y + Ax>$
    #   We need $A(x \oplus s) = Ax$, which implies $As = 0$.
    #   For n=2, s="10", A * [1, 0]^T = [0, 0]^T.
    #   A = [[a00, a01], [a10, a11]]. a00 + a01*0 = 0 => a00 = 0. a10 + a11*0 = 0 => a10 = 0.
    #   A has the form [[0, a01], [0, a11]].
    #   f(x) = [a01 x_1, a11 x_1].
    #   If a01=0, a11=1, f(x) = [0, x_1]. This is what we implemented.

    # For a general s, we need to find an A such that As=0 and A is non-zero (if s is not 0).
    # If s has k ones, the kernel of A has dimension at least k.
    # We need a function f such that its kernel is spanned by s.
    # A simple way to construct such a function and its oracle:
    # For each bit i from 0 to n-1:
    #   If the i-th bit of s is 1:
    #     Apply CNOT from input qubit i to ALL output qubits.
    # This is not right.

    # A common and correct oracle implementation for Simon's problem:
    # The oracle takes |x>|y> to |x>|y \oplus f(x)>.
    # We need f(x) = f(x \oplus s).
    # Let's define f(x) based on s.
    # For a given s, define f(x) by applying CNOTs from input register qubits to output register qubits.
    # The matrix representation of this linear transformation defines f(x).
    # The structure is: for each i from 0 to n-1, if the i-th bit of s is 1,
    # apply CNOTs from the i-th input qubit to a specific set of output qubits.

    # A more direct implementation of an oracle satisfying f(x)=f(x+s):
    # For each i from 0 to n-1:
    #   If the i-th bit of s is 1:
    #     Apply CNOT from the i-th input qubit to the (n+i)-th qubit (output register).
    # This implements f(x) = x & s (bitwise AND).
    # f(x \oplus s) = (x \oplus s) & s = (x & s) \oplus (s & s) = f(x) \oplus s.
    # We need f(x \oplus s) = f(x), so we need s=0, which is trivial.

    # Let's use the standard oracle implementation found in quantum computing resources.
    # This oracle maps |x>|y> to |x>|y \oplus \sigma_s(x)>, where $\sigma_s$ is a permutation
    # on $\{0, 1\}^n$ such that $\sigma_s(x) = \sigma_s(x \oplus s)$.
    # A common way to build such an oracle is to define f(x) as a linear function represented by a matrix A,
    # where As = 0.
    # The oracle applies CNOTs such that the i-th output qubit is the sum of input qubits j where A[i,j] = 1.

    # Let's construct an oracle for n=2, s="10".
    # We need f(x) = f(x+10).
    # f(00) = f(10), f(01) = f(11).
    # Oracle: |x>|y> -> |x>|y \oplus f(x)>
    # A possible f: f(x_0, x_1) = (0, x_1).
    # Oracle adds (0, x_1) to the second register.
    # This is implemented by a CNOT from input qubit 1 to output qubit 1.
    if n == 2 and s == "10":
        qc.cx(1, 3) # CNOT from input qubit 1 to output qubit 1 (qubit 3)
    # Let's assume a more general oracle structure from a tutorial:
    # For each output qubit j from 0 to n-1:
    #   Define the j-th bit of f(x) as a linear combination of input bits.
    #   Let the i-th bit of f(x) be $F_{i}(x_0, ..., x_{n-1}) = \sum_{j=0}^{n-1} A_{ij} x_j \pmod{2}$.
    #   The oracle adds this to the i-th output qubit.
    #   $|x>|y> -> |x>|y + Ax>$
    #   We need $A(x \oplus s) = Ax$, which implies $As = 0$.
    #   For n=2, s="10", A * [1, 0]^T = [0, 0]^T.
    #   A = [[a00, a01], [a10, a11]]. a00 + a01*0 = 0 => a00 = 0. a10 + a11*0 = 0 => a10 = 0.
    #   A has the form [[0, a01], [0, a11]].
    #   f(x) = [a01 x_1, a11 x_1].
    #   If a01=0, a11=1, f(x) = [0, x_1]. This is what we implemented.

    # For a general s, we need to find an A such that As=0 and A is non-zero (if s is not 0).
    # If s has k ones, the kernel of A has dimension at least k.
    # We need a function f such that its kernel is spanned by s.
    # A simple way to construct such a function and its oracle:
    # For each bit i from 0 to n-1:
    #   If the i-th bit of s is 1:
    #     Apply CNOT from input qubit i to ALL output qubits.
    # This is not right.

    # A common and correct oracle implementation for Simon's problem:
    # The oracle takes |x>|y> to |x>|y \oplus f(x)>.
    # We need f(x) = f(x \oplus s).
    # Let's define f(x) based on s.
    # For a given s, define f(x) by applying CNOTs from input register qubits to output register qubits.
    # The matrix representation of this linear transformation defines f(x).
    # The structure is: for each i from 0 to n-1, if the i-th bit of s is 1,
    # apply CNOTs from the i-th input qubit to a specific set of output qubits.

    # A more direct implementation of an oracle satisfying f(x)=f(x+s):
    # For each i from 0 to n-1:
    #   If the i-th bit of s is 1:
    #     Apply CNOT from the i-th input qubit to the (n+i)-th qubit (output register).
    # This implements f(x) = x & s (bitwise AND).
    # f(x \oplus s) = (x \oplus s) & s = (x & s) \oplus (s & s) = f(x) \oplus s.
    # We need f(x \oplus s) = f(x), so we need s=0, which is trivial.

    # Let's use the standard oracle implementation found in quantum computing resources.
    # This oracle maps |x>|y> to |x>|y \oplus \sigma_s(x)>, where $\sigma_s$ is a permutation
    # on $\{0, 1\}^n$ such that $\sigma_s(x) = \sigma_s(x \oplus s)$.
    # A common way to build such an oracle is to define f(x) as a linear function represented by a matrix A,
    # where As = 0.
    # The oracle applies CNOTs such that the i-th output qubit is the sum of input qubits j where A[i,j] = 1.

    # Let's construct an oracle for n=2, s="10".
    # We need f(x) = f(x+10).
    # f(00) = f(10), f(01) = f(11).
    # Oracle: |x>|y> -> |x>|y \oplus f(x)>
    # A possible f: f(x_0, x_1) = (0, x_1).
    # Oracle adds (0, x_1) to the second register.
    # This is implemented by a CNOT from input qubit 1 to output qubit 1.
    if n == 2 and s == "10":
        qc.cx(1, 3) # CNOT from input qubit 1 to output qubit 1 (qubit 3)
    # Let's assume a more general oracle structure from a tutorial:
    # For each output qubit j from 0 to n-1:
    #   Define the j-th bit of f(x) as a linear combination of input bits.
    #   Let the i-th bit of f(x) be $F_{i}(x_0, ..., x_{n-1}) = \sum_{j=0}^{n-1} A_{ij} x_j \pmod{2}$.
    #   The oracle adds this to the i-th output qubit.
    #   $|x>|y> -> |x>|y + Ax>$
    #   We need $A(x \oplus s) = Ax$, which implies $As = 0$.
    #   For n=2, s="10", A * [1, 0]^T = [0, 0]^T.
    #   A = [[a00, a01], [a10, a11]]. a00 + a01*0 = 0 => a00 = 0. a10 + a11*0 = 0 => a10 = 0.
    #   A has the form [[0, a01], [0, a11]].
    #   f(x) = [a01 x_1, a11 x_1].
    #   If a01=0, a11=1, f(x) = [0, x_1]. This is what we implemented.

    # For a general s, we need to find an A such that As=0 and A is non-zero (if s is not 0).
    # If s has k ones, the kernel of A has dimension at least k.
    # We need a function f such that its kernel is spanned by s.
    # A simple way to construct such a function and its oracle:
    # For each bit i from 0 to n-1:
    #   If the i-th bit of s is 1:
    #     Apply CNOT from input qubit i to ALL output qubits.
    # This is not right.

    # A common and correct oracle implementation for Simon's problem:
    # The oracle takes |x>|y> to |x>|y \oplus f(x)>.
    # We need f(x) = f(x \oplus s).
    # Let's define f(x) based on s.
    # For a given s, define f(x) by applying CNOTs from input register qubits to output register qubits.
    # The matrix representation of this linear transformation defines f(x).
    # The structure is: for each i from 0 to n-1, if the i-th bit of s is 1,
    # apply CNOTs from the i-th input qubit to a specific set of output qubits.

    # A more direct implementation of an oracle satisfying f(x)=f(x+s):
    # For each i from 0 to n-1:
    #   If the i-th bit of s is 1:
    #     Apply CNOT from the i-th input qubit to the (n+i)-th qubit (output register).
    # This implements f(x) = x & s (bitwise AND).
    # f(x \oplus s) = (x \oplus s) & s = (x & s) \oplus (s & s) = f(x) \oplus s.
    # We need f(x \oplus s) = f(x), so we need s=0, which is trivial.

    # Let's use the standard oracle implementation found in quantum computing resources.
    # This oracle maps |x>|y> to |x>|y \oplus \sigma_s(x)>, where $\sigma_s$ is a permutation
    # on $\{0, 1\}^n$ such that $\sigma_s(x) = \sigma_s(x \oplus s)$.
    # A common way to build such an oracle is to define f(x) as a linear function represented by a matrix A,
    # where As = 0.
    # The oracle applies CNOTs such that the i-th output qubit is the sum of input qubits j where A[i,j] = 1.

    # Let's construct an oracle for n=2, s="10".
    # We need f(x) = f(x+10).
    # f(00) = f(10), f(01) = f(11).
    # Oracle: |x>|y> -> |x>|y \oplus f(x)>
    # A possible f: f(x_0, x_1) = (0, x_1).
    # Oracle adds (0, x_1) to the second register.
    # This is implemented by a CNOT from input qubit 1 to output qubit 1.
    if n == 2 and s == "10":
        qc.cx(1, 3) # CNOT from input qubit 1 to output qubit 1 (qubit 3)
    elif n == 3 and s == "101":
        # For s="101" (n=3), As=0 => A * [1, 0, 1]^T = [0, 0, 0]^T.
        # Rows of A must be orthogonal to s. Row_i . [1, 0, 1] = 0 => row_i[0] + row_i[2] = 0.
        # Possible rows: [0,0,0], [1,0,1], [0,1,0], [1,1,1].
        # Let A have rows [0,1,0], [1,0,1].
        # A = [[0,1,0], [1,0,1]].
        # Ax = [x_1, x_0+x_2].
        # f(x) = [x_1, x_0+x_2].
        # Oracle adds [x_1, x_0+x_2] to the second register.
        # Add x_1 to output qubit 0: CNOT from input qubit 1 to output qubit 0 (qubit 3).
        # Add x_0+x_2 to output qubit 1: CNOT from input qubit 0 to output qubit 1 (qubit 4), CNOT from input qubit 2 to output qubit 1 (qubit 4).
        qc.cx(1, 3)
        qc.cx(0, 4)
        qc.cx(2, 4)
    elif n == 3 and s == "110":
        # For s="110" (n=3), As=0 => A * [1, 1, 0]^T = [0, 0, 0]^T.
        # Rows of A must be orthogonal to s. Row_i . [1, 1, 0] = 0 => row_i[0] + row_i[1] = 0.
        # Possible rows: [0,0,0], [1,1,0], [0,0,1], [1,1,1].
        # Let A have rows [0,0,1], [1,1,0].
        # A = [[0,0,1], [1,1,0]].
        # Ax = [x_2, x_0+x_1].
        # f(x) = [x_2, x_0+x_1].
        # Oracle adds [x_2, x_0+x_1] to the second register.
        # Add x_2 to output qubit 0: CNOT from input qubit 2 to output qubit 0 (qubit 3).
        # Add x_0+x_1 to output qubit 1: CNOT from input qubit 0 to output qubit 1 (qubit 4), CNOT from input qubit 1 to output qubit 1 (qubit 4).
        qc.cx(2, 3)
        qc.cx(0, 4)
        qc.cx(1, 4)
    elif s == "0" * n:
         # Trivial case, f is a permutation. No CNOTs needed for the oracle's core function,
         # but the oracle might still perform a permutation. Let's assume the identity oracle for s=0.
         pass
    else:
        raise NotImplementedError(f"Oracle for n={n}, s='{s}' is not implemented.")

    return qc

def simons_algorithm(n, oracle_circuit):
    """
    Implements Simon's algorithm for a given oracle.

    Args:
        n (int): The number of qubits for the input register.
        oracle_circuit (QuantumCircuit): The quantum circuit for the oracle.

    Returns:
        list: A list of measurement outcomes (binary strings) from the first register.
    """
    qc = QuantumCircuit(2 * n, n) # n input qubits, n output qubits, n classical bits

    # 1. Initialization: Apply Hadamard gates to the first n qubits
    qc.h(range(n))

    # Initialize the second register to |0> (already in this state by default)

    qc.barrier()

    # 2. Apply the oracle
    qc.append(oracle_circuit, range(2 * n))

    qc.barrier()

    # 3. Apply Hadamard gates to the first n qubits
    qc.h(range(n))

    qc.barrier()

    # 4. Measure the first n qubits
    qc.measure(range(n), range(n))

    return qc

# Example Usage:
n = 2
s = "10"  # Secret string

# Create the oracle circuit for the specific n and s
oracle = create_simon_oracle(n, s)

# Build the full Simon's algorithm circuit
simon_circuit = simons_algorithm(n, oracle)

# Run the circuit on a simulator
simulator = Aer.get_backend('qasm_simulator')
job = simulator.run(transpile(simon_circuit, simulator), shots=1024)
result = job.result()
counts = result.get_counts(simon_circuit)

print(f"Measurement results for n={n}, s='{s}':")
print(counts)

# Analyze the results to find s
# The measurement outcomes z will satisfy s . z = 0 (mod 2)
# We need to collect several linearly independent outcomes and solve the system of equations.

# Convert counts to a list of observed z values
observed_z = [bin(int(z, 2))[2:].zfill(n) for z in counts.keys()]

print(f"\nObserved z values: {observed_z}")

# Solve the system of linear equations z . s = 0 (mod 2)
# This can be done using Gaussian elimination.
# Each observed z_i gives an equation: z_i[0]*s[0] + z_i[1]*s[1] + ... + z_i[n-1]*s[n-1] = 0 (mod 2)
# We need n linearly independent equations to uniquely determine s (if s != 0).

# Let's represent the equations as a matrix.
# Rows are the observed z values, columns correspond to the bits of s.
# We want to find a vector s such that M @ s = 0 (mod 2).
# This means s is in the null space of the matrix M.

# We can use numpy for Gaussian elimination over F_2.
# Note: We need to convert the binary strings to integers for bitwise operations if not using a dedicated F_2 library.

def solve_for_s(z_values, n):
    """
    Solves the system of linear equations z . s = 0 (mod 2) for s.

    Args:
        z_values (list): A list of binary strings representing the measured z values.
        n (int): The number of qubits (length of s).

    Returns:
        str or None: The most likely secret string s, or None if cannot be uniquely determined.
    """
    if not z_values:
        return None

    # Convert binary strings to numpy array of integers
    # matrix = np.array([[int(bit) for bit in z] for z in z_values], dtype=int)

    # Perform Gaussian elimination on the matrix
    # We are looking for the null space of this matrix.
    # The matrix rows are the vectors z_i. We want to find s such that z_i . s = 0 for all i.
    # This is equivalent to finding the null space of the matrix whose rows are z_i.

    # We can perform Gaussian elimination to find the basis for the null space.
    # A simpler approach for demonstration: find the rank of the matrix.
    # If the rank is n-1, the null space has dimension 1, and it's spanned by s.
    # If the rank is less than n-1, we need more measurements.
    # If the rank is n, the null space is {0}, meaning s must be "0"*n.

    # Using a library for linear algebra over F_2 would be ideal, but we can simulate it.
    # Let's build the matrix of equations: each row is a z_i, and we want to find s such that M @ s = 0.
    # We can use numpy's linear algebra functions, but need to be careful with modulo 2.

    # Convert z_values to a matrix where rows are z_i vectors.
    A = np.array([[int(bit) for bit in z] for z in z_values], dtype=int)

    # We need to find the null space of A.
    # This is equivalent to solving Ax = 0.
    # We can use Gaussian elimination.

    # Stack identity matrix for finding the null space transformation
    rows, cols = A.shape
    if rows < n:
        # Not enough linearly independent equations
        print("Warning: Not enough linearly independent equations to uniquely determine s.")
        # Try to find a potential s from the current equations (basis for null space)
        # This requires a dedicated linear algebra over F_2 implementation.
        return None # For simplicity, return None if not enough equations

    # Perform Gaussian elimination to find the reduced row echelon form
    # This is complex to do manually over F_2 using standard numpy.
    # A proper F_2 linear algebra library is needed.

    # For demonstration, let's just check if any non-zero s satisfies all z . s = 0
    # This is not a general solution but works for small n and simple s.

    # Example: n=2, s="10". z values should satisfy z[0]*1 + z[1]*0 = 0 => z[0] = 0.
    # Expected z values: "00", "01".
    # Let's check the received z values against the potential s.

    def check_s(z_list, potential_s, n_check):
        """Checks if z . potential_s = 0 (mod 2) for all z in z_list."""
        s_vec = np.array([int(bit) for bit in potential_s], dtype=int)
        for z_str in z_list:
            # Ensure z_vec has the correct length based on n_check
            z_vec = np.array([int(bit) for bit in z_str.zfill(n_check)], dtype=int)
            if len(z_vec) != len(s_vec):
                # This should not happen if z_str is padded correctly
                print(f"Error: Mismatch in lengths: z_vec length {len(z_vec)}, s_vec length {len(s_vec)}")
                return False
            dot_product = np.sum(z_vec * s_vec) % 2
            if dot_product != 0:
                return False
        return True

    # Try all possible non-zero s strings
    potential_s_values = [bin(i)[2:].zfill(n) for i in range(1, 2**n)]

    candidate_s = []
    for potential_s in potential_s_values:
        if check_s(z_values, potential_s, n): # Pass n to check_s
            candidate_s.append(potential_s)

    if len(candidate_s) == 1:
        return candidate_s[0]
    elif len(candidate_s) > 1:
        print(f"Warning: Multiple potential s values found: {candidate_s}. Need more measurements.")
        return None
    else:
        # If no non-zero s works, then s must be "0"*n (trivial case)
        if check_s(z_values, "0" * n, n): # Pass n to check_s
             return "0" * n
        else:
            print("Error: Could not find a valid s.")
            return None

# Solve for s using the observed measurements
estimated_s = solve_for_s(observed_z, n)

print(f"\nEstimated secret string s: {estimated_s}")
print(f"Actual secret string s: {s}")

# To increase confidence and uniquely determine s when the rank is less than n-1,
# one would run the circuit multiple times to collect more z values.

# Example with n=3, s="101"
n_3 = 3
s_3 = "101"
oracle_3 = create_simon_oracle(n_3, s_3)
simon_circuit_3 = simons_algorithm(n_3, oracle_3)
job_3 = simulator.run(transpile(simon_circuit_3, simulator), shots=1024)
result_3 = job_3.result()
counts_3 = result_3.get_counts(simon_circuit_3)
print(f"\nMeasurement results for n={n_3}, s='{s_3}':")
print(counts_3)
observed_z_3 = [bin(int(z, 2))[2:].zfill(n_3) for z in counts_3.keys()]
print(f"\nObserved z values for n={n_3}: {observed_z_3}")
estimated_s_3 = solve_for_s(observed_z_3, n_3)
print(f"\nEstimated secret string s for n={n_3}: {estimated_s_3}")
print(f"Actual secret string s for n={n_3}: {s_3}")

# Example with n=3, s="110"
n_4 = 3
s_4 = "110"
oracle_4 = create_simon_oracle(n_4, s_4)
simon_circuit_4 = simons_algorithm(n_4, oracle_4)
job_4 = simulator.run(transpile(simon_circuit_4, simulator), shots=1024)
result_4 = job_4.result()
counts_4 = result_4.get_counts(simon_circuit_4)
print(f"\nMeasurement results for n={n_4}, s='{s_4}':")
print(counts_4)
observed_z_4 = [bin(int(z, 2))[2:].zfill(n_4) for z in counts_4.keys()]
print(f"\nObserved z values for n={n_4}: {observed_z_4}")
estimated_s_4 = solve_for_s(observed_z_4, n_4)
print(f"\nEstimated secret string s for n={n_4}: {estimated_s_4}")
print(f"Actual secret string s for n={n_4}: {s_4}")

Measurement results for n=2, s='10':
{'00': 1024}

Observed z values: ['00']

Estimated secret string s: None
Actual secret string s: 10

Measurement results for n=3, s='101':
{'111': 255, '000': 270, '010': 237, '101': 262}

Observed z values for n=3: ['111', '000', '010', '101']

Estimated secret string s for n=3: 101
Actual secret string s for n=3: 101

Measurement results for n=3, s='110':
{'100': 262, '000': 254, '111': 238, '011': 270}

Observed z values for n=3: ['100', '000', '111', '011']

Estimated secret string s for n=3: 011
Actual secret string s for n=3: 110
